In [2]:
import os
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import accuracy, NaiveBayesClassifier
from string import punctuation
import pickle

MODEL_PATH = './model.pickle'
DATA_PATH = './financial_dataset.csv'

# Training Model

## Load

In [16]:
dframe = pd.read_csv(DATA_PATH)
x = dframe['Statement']
y = dframe['Sentiment']

statement = ' '.join(x)
statement

'The GeoSolutions technology will leverage Benefon \'s GPS solutions by providing Location Based Search Technology , a Communities Platform , location relevant multimedia content and a new and powerful commercial model . $ESI on lows, down $1.50 to $2.50 BK a real possibility For the last quarter of 2010 , Componenta \'s net sales doubled to EUR131m from EUR76m for the same period a year earlier , while it moved to a zero pre-tax profit from a pre-tax loss of EUR7m . $SPY wouldn\'t be surprised to see a green close Shell\'s $70 Billion BG Deal Meets Shareholder Skepticism SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANGE RELEASE OCTOBER 14 , 2008 AT 2:45 PM The Company updates its full year outlook and estimates its results to remain at loss for the full year . Kone \'s net sales rose by some 14 % year-on-year in the first nine months of 2008 . Circulation revenue has increased by 5 % in Finland and 4 % in Sweden in 2008 . $SAP Q1 disappoints as #software licenses down. Real problem? #Cl

## Preprocessing

In [36]:
eng_stop = stopwords.words('english')       

In [34]:
def altertag (tag: str):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('R'):
        return 'r'
    else:
        return 'n'
         
def lemmatizer(tokens):
    res = []
    tagged = pos_tag(tokens)
    
    for word, tag in tagged:
        temp = WordNetLemmatizer().lemmatize(word, altertag(tag))
        res.append(temp)
    return res
    

def preprocesser(state):
    state = state.lower()
    tokens = word_tokenize(state)
    tokens = [tok for tok in tokens if tok not in eng_stop]
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok.isalpha()]
    tokens = lemmatizer(tokens)
    return tokens

    

In [38]:
tokens = preprocesser(statement)
freq = FreqDist(tokens)
freq

FreqDist({'eur': 747, 'mn': 459, 'profit': 368, 'sale': 338, 'company': 321, 'net': 297, 'say': 296, 'finnish': 266, 'million': 243, 'year': 234, ...})

## Feature Extraction

In [41]:
def extract_feat(state):
    feat = {}
    for tok in freq.keys():
        feat[tok] = (tok in state)
    return feat

In [45]:
featlist = [(extract_feat(preprocesser(state)), ans)
            for state, ans in zip(x, y)]

## Training

In [46]:
split = int(len(featlist)*0.8)
train = featlist[:split]
eval  = featlist[split:]

# Menu

## Support Function

In [ ]:
currstate = ''

In [ ]:
def WriteState():
    state = 'abc'
    
    while True:
        print()
        state = input('Input Statement: ')
        
        if len(state.split(' ')) >= 2:
            break
        else:
            print('(!) Please write at least 2 word')
    
    # Model
    currstate = state
    if (os.path.exists(MODEL_PATH)):
        with open(MODEL_PATH, 'rb') as f:
            model = pickle.load(f)
    else:
        model = training()
    
    # Result
    

def training():
    model = NaiveBayesClassifier.train(train)
    model.show_most_informative_features(7)
    
    acc = accuracy(eval)
    print(f'Accuracy: {accuracy*100}%')
    
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(model, f)
        
    return model

In [ ]:
def AnalyState():
    if len(currstate) == 0:
        print('Please Input your Statement first')
        return

def ShowPostag():
    print('Pos Tag')
    tokens = preprocesser(currstate)
    tagged = pos_tag(tokens)
    
    for word, tag in tagged:
        
    

## Main Program

In [50]:
cc = -1

while True:
    print()
    print('1. Write your Statement')
    print('2. Analyse your Statement')
    print('3. Exit')
    cc = input('>> ')
    
    try:
       cc = int(cc)
    except:
        print('(!) Expected Integer as an input')
        continue
    
    if cc == 1:
        # 1
        WriteState()
    elif cc == 2:
        # 2
        pass
    elif cc == 3:
        break
    else:
        print('(!) Invalid Choice')
     
    


1. Write your Statement
2. Analyse your Statement
3. Exit



TypeError: NaiveBayesClassifier.__init__() missing 1 required positional argument: 'feature_probdist'